In [1]:
import torch
import torch.nn as nn

c:\Users\Manoj\Python ML\Lib\site-packages\torch\cuda\__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
class MultiHeadLatentAttention(nn.Module):
  def __init__(self, d_model, context_length, num_heads, d_latent, dropout=0.0):
    super().__init__()
    # d_in = d_out = d_model
    assert d_model % num_heads == 0,  "d_model must be divide by num_heads"

    self.d_model = d_model
    self.num_heads = num_heads 
    self.head_dim = d_model // num_heads
    self.d_latent = d_latent 

    # Query
    self.W_q = nn.Linear(d_model, d_model, bias=False)

    # KV Down-Projector  # compress
    self.W_dkv = nn.Linear(d_model, d_latent, bias=False)

    # The new Key and Value Up-Projectors. # decompress
    self.W_uk = nn.Linear(d_latent, d_model, bias=False)
    self.W_uv = nn.Linear(d_latent, d_model, bias=False)

    # The Final output projection
    self.W_o = nn.Linear(d_model, d_model)

    self.dropout = nn.Dropout(dropout)
    # Causal mask to prevent attending to future tokens.
    self.register_buffer('mask', torch.triu(torch.ones(1, 1, context_length, context_length), diagonal=1).bool())


  def forward(self, x):
    b, num_tokens, d_model = x.shape

    # 1. Query Path (Unchanged)
    # Project and reshape the query as in standard MHA.
    q = self.W_q(x) # (b, num_tokens, d_model)
    q = q.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)  # (b, num_heads num_tokens, head_dim)

    # 2. Key/Value Path (The MLA Innovation)
    # Step 2a: Down-Project to the latent space.
    # This is the ONLY value that would be cached during inference.
    c_kv = self.W_dkv(x)  # (b, num_tokens, d_latent)

    # Step 2b: Up-Project from the latent space to get full K and V.
    # These are computed on the fly and are not cached.
    k = self.W_uk(c_kv).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)#(b, num_heads num_tokens, head_dim)
    v = self.W_uv(c_kv).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)#(b, num_heads num_tokens, head_dim)

    # 3. Standard Attention Calculation
    # The rest of the process is identical to standard MHA.
    attn_scores = q @ k.transpose(2, 3)
    attn_scores = attn_scores.masked_fill(
        self.mask[:, :, :num_tokens, :num_tokens], float('-inf'))

    attn_weights = torch.softmax(attn_scores / k.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vector = (attn_weights @ v).transpose(1, 2).contiguous().view(b, num_tokens, self.d_model)

    # 4. Final Output Projection
    output = self.W_o(context_vector)
    return output


In [3]:
# --- Usage Example ---
d_in, d_out = 512, 512
d_model = 512
num_heads = 8
d_latent = 128  # Latent dimension must be smaller than d_model
batch_size = 4
context_length=64

# Instantiate the layer
# mla_layer = MultiHeadLatentAttention(d_in, d_out, context_length, num_heads, d_latent, dropout=0.0)
mla_layer = MultiHeadLatentAttention(d_model, context_length, num_heads, d_latent, dropout=0.0)

# Create a dummy input tensor
dummy_input = torch.randn(batch_size, context_length, d_in)

# Pass the input through the layer
output = mla_layer(dummy_input)

print("MLA Layer successful!")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

MLA Layer successful!
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])
